# QDC Emulation Framework — Main Notebook

This notebook reproduces all experimental results presented in:

> **"A Framework for Quantum Data Center Emulation Using Digital Quantum Computers"**  
> S. N. Elyasi, P. Monti, J. Li, and R. Lin — Chalmers University of Technology / Soochow University

Each section corresponds to a figure in the paper.  Hardware results were obtained on IBM's
**ibm_torino** (133-qubit Heron R1 processor) using 10 000 measurement shots per data point.
Numerical reference curves use Qiskit's `AerSimulator` configured with a backend-derived noise model.

---


## 0 · Imports and Dependencies

All distributed-QC primitives live in the `QdcEm` package:

| Module | Contents |
|---|---|
| `Algorithms` | Monolithic & distributed circuits (Grover, QFT) and CM unitary |
| `RemoteGates` | Noisy remote gate primitives (Cat-Comm, TP1) |
| `QPU` | QPU layout helper and `Get_Initial_Layout` |
| `Representation` | Publication-quality plotting functions |


In [ ]:
from QdcEm import Algorithms, RemoteGates, QPU, Representation

from math import pi, sqrt, exp
from collections import defaultdict
import time

import numpy as np
import matplotlib.pyplot as plt

from qutip import *

from qiskit import *
from qiskit_aer import AerSimulator
from qiskit.transpiler import Layout
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2


## 1 · Global Experimental Parameters

These values are held constant across **all** experiments (Sections 3.A–3.D of the paper):

| Parameter | Value | Description |
|---|---|---|
| `kappa_Transductor` | 0.5 | Transducer coupling constant κ_T |
| `steps` | 10 | Maximum number of 10 m fiber segments |
| `shots` | 10 000 | Measurement shots per data point |

κ_T = 0.5 was chosen empirically to produce an ~30 % initial drop in Bell-pair
fidelity while still allowing the DQC circuits to execute successfully (Section 2.C).


In [ ]:
steps             = 10        # fiber-step data points (each step ≡ 10 m)
kappa_Transductor = 0.5       # transducer coupling constant κ_T (paper Sec. 2.C)
shots             = 10_000    # measurement shots per circuit execution


## 2 · IBM Quantum Backend

All hardware experiments use **ibm_torino** (133-qubit Heron R1 processor, Fig. 4 of the paper).
Circuits are transpiled with Qiskit's preset pass-manager at optimisation level 3 using the
native gate set {CZ, I, RZ, SX, X}.

> ⚠️  Replace the token with your own IBM Quantum credentials before submitting to hardware.  
> Set `simulator = True` in Section 5 to redirect all jobs to `AerSimulator` locally.


In [ ]:
import json, pathlib

# Load credentials from ibm_credentials.json (sits next to this notebook)
_cred_path = pathlib.Path("ibm_credentials.json")
if _cred_path.exists():
    _creds = json.loads(_cred_path.read_text())
    service = QiskitRuntimeService(
        channel =_creds["channel"],
        instance=_creds["instance"],
        token   =_creds["token"],
    )
else:
    # Fallback: hard-code credentials here
    service = QiskitRuntimeService(
        channel ='ibm_quantum_platform',
        instance='crn:v1:bluemix:public:quantum-computing:us-east:'
                 'a/c9a9b9d7c5bd452daa3229a9a4dc4056:'
                 'd3802968-f622-4b53-94c7-f99f2ccb081b::',
        token   ='<YOUR_IBM_QUANTUM_TOKEN>',
    )

backend = service.backend("ibm_torino")
print(f"Backend : {backend.name}")
print(f"Qubits  : {backend.num_qubits}")


## 3 · Fiber Attenuation Calibration

Following Section 2.C of the paper, the fiber coupling constant κ_F is derived from the
physical attenuation coefficient α (km⁻¹) by imposing D(1) = 0.01 km (one collision ≡ 10 m):

$$\kappa_F = \sqrt{0.01 \cdot \alpha}$$

Three standard telecom fiber types are used, plus a numerical (AerSimulator) reference.

![Communication channel noise model](Pics/ModelNoise.png)


In [ ]:
# Physical attenuation coefficients α (km⁻¹) — used for display in legends
fibers_alpha = {
    'G-652-D'  : 0.0415,
    'G-654-E'  : 0.0392,   # lowest attenuation — used for QFT and Grover
    'G-655-D'  : 0.0507,
    'Numerical': 0.0415,   # AerSimulator reference, same α as G-652-D
}

d = 0.01   # one collision ≡ 10 m = 0.01 km  (Section 2.C)

# κ_F = sqrt(d · α)
fibers = {name: np.sqrt(d * alpha) for name, alpha in fibers_alpha.items()}

print("Fiber coupling constants κ_F (used in circuit construction):")
for name, kF in fibers.items():
    print(f"  {name:12s}  κ_F = {kF:.6f}   (α = {fibers_alpha[name]:.4f} km⁻¹)")


## 4 · QPU Partitioning on ibm_torino

The ibm_torino coupling map is partitioned into two logical QPUs by selecting qubits
whose boundary communication qubits share a **direct edge** (dist_G = 1) in the coupling
graph, as required by Algorithm 1 of the paper.

The selected partition — shown in the figure below — uses qubits from the **top-left**
corner of the device, which typically exhibit the best calibration (lowest gate error
and longest coherence times T₁, T₂):

```
QPU A  (purple box, left)          QPU B  (purple box, right)
─────────────────────────          ──────────────────────────
  qubit 0  ─ qubit 1 ─ qubit 2 ─ qubit 3 ═══ qubit 4 ─ qubit 5 ─ qubit 6 ─ qubit 7
  qubit 14 (hanging)                          qubit 2 (hanging)
```

The **boundary edge 3 ↔ 4** is the direct qubit-to-qubit coupling that acts as the
quantum communication channel between QPU A and QPU B (Fig. 3b of the paper).

Role assignments inside each QPU:

| Role | QPU A qubit | QPU B qubit | Rationale |
|---|---|---|---|
| **Processing** | 0 | 7 | Endpoint qubits, away from boundary |
| **Communication** | 3 | 4 | Directly coupled — the inter-QPU quantum channel |
| **Environment (ENV)** | 14 | 2 | Hanging qubits; recycled via Reset for CM noise |

`QPU.Get_Initial_Layout` maps the abstract quantum register to physical qubits in the
order: `Comm → EN → Processing_Qubits` for each QPU in sequence.  For the standard
6-qubit remote-gate register `q[0..5]` this gives:

| Register qubit | Physical qubit | Role |
|---|---|---|
| `q[0]` | **3** | CommA (communication, QPU A) |
| `q[1]` | **14** | ENA (environment, QPU A) |
| `q[2]` | **0** | Processing A |
| `q[3]` | **4** | CommB (communication, QPU B) |
| `q[4]` | **2** | ENB (environment, QPU B) |
| `q[5]` | **7** | Processing B |

![ibm_torino qubit coupling map with QPU A and QPU B highlighted](Pics/Mapping.png)


In [ ]:
# ── Physical qubit assignments ────────────────────────────────────────────────
#
# QPU A  (left purple box in coupling-map figure)
#   Processing qubit : 0   (left end of QPU A row, away from boundary)
#   Communication    : 3   (rightmost qubit of QPU A — directly coupled to qubit 4)
#   Environment      : 14  (hanging qubit below qubit 0, recycled via Reset)
#
# QPU B  (right purple box in coupling-map figure)
#   Communication    : 4   (leftmost qubit of QPU B — directly coupled to qubit 3)
#   Environment      : 2  (hanging qubit below qubit 4, recycled via Reset)
#   Processing qubit : 7   (right end of QPU B row, away from boundary)
#
# Boundary edge  :  3 ↔ 4  (direct coupling — the inter-QPU quantum channel)

QPUA_layout = QPU.Make(Comm=3,  EN=14, Processing_Qubits=[0])
QPUB_layout = QPU.Make(Comm=4,  EN= 2, Processing_Qubits=[7])

QPUs = [QPUA_layout, QPUB_layout]

# Verify layout mapping produced by Get_Initial_Layout
# For a 6-qubit register q[0..5]:
#   q[0] → QPUA.Comm = 3
#   q[1] → QPUA.EN   = 14
#   q[2] → QPUA.Processing[0] = 0
#   q[3] → QPUB.Comm = 4
#   q[4] → QPUB.EN   = 2
#   q[5] → QPUB.Processing[0] = 7
_q_test = QuantumRegister(6, name='q')
_layout = QPU.Get_Initial_Layout(QPUs, QRG=_q_test)
print("Initial layout (register qubit → physical qubit):")
for reg_q, phys_q in sorted(_layout.get_physical_bits().items()):
    print(f"  physical {phys_q:3d}  ←  register q[{list(_q_test).index(reg_q) if reg_q in list(_q_test) else '?'}]")


### Choosing the qubit roles — and why the heavy-hex layout constrains them

The role of every physical qubit in the partition above is dictated by **Algorithm 1's boundary constraints**, not chosen freely:

- **Communication qubits — `3` (QPU A) and `4` (QPU B).** These are the only qubits that form the inter-QPU channel. They sit on opposite sides of the partition boundary and are joined by a **direct edge** in the coupling graph (`dist_G(3, 4) = 1`). The boundary rule `dist_G ∈ {1, ∞}` forbids any multi-hop path between QPUs, so the two communication qubits must be genuine nearest neighbours — edge **`3 ↔ 4`** is exactly the direct qubit-to-qubit coupling that stands in for the optical link (Fig. 3b of the paper).
- **Processing qubits — `0` (QPU A) and `7` (QPU B).** Placed at the *far ends* of each QPU row, as far from the boundary as the partition allows, so the local algorithm logic never overlaps with the communication/environment machinery.
- **Environment qubits — `14` (QPU A) and `2` (QPU B).** Auxiliary ancillas that implement the collision model. Each is a free *"hanging"* qubit next to its QPU: physical **`14`** sits below processing qubit `0` in QPU A, and physical **`2`** sits below communication qubit `4` in QPU B. They hold no algorithmic state — each is **reset and recycled** after every collision step (Sec. 3.B), so a single spare neighbour per QPU is sufficient. The asymmetry of the two choices (one beside the processing qubit, one beside the communication qubit) is itself a symptom of the connectivity constraint described next.


**Why these specific, slightly awkward placements?** `ibm_torino` is a **Heron r1** device with a **heavy-hexagonal** coupling map. In a heavy-hex lattice qubits have at most **degree 3**, and many have only **degree 2** — the connectivity is deliberately sparse to suppress crosstalk and frequency collisions. That sparsity is the binding constraint: for each communication qubit there are very few free neighbours that can host an environment ancilla *and* stay clear of the processing/communication path, which is why the environment qubits land on irregular hanging sites (`14`, `2`) instead of clean, symmetric positions. The same sparsity limits how the two partitions can be drawn while keeping their communication qubits directly coupled.


**Nighthawk relaxes this constraint.** IBM's **Nighthawk** architecture — introduced *after* this paper was written, and delivered to users at the end of 2025 — abandons the heavy-hex layout for a **dense square lattice** in which each of its 120 qubits couples to its **four nearest neighbours** through 218 tunable couplers (over 20% more connectivity than Heron). With degree-4 connectivity, every communication qubit has several free neighbours available to host an environment ancilla, and partition boundaries can be drawn far more freely — so the same framework maps onto Nighthawk-class hardware much more naturally, with more room to place environment qubits and to scale beyond two QPUs. The figure below shows two example QPU partitions on the Nighthawk square lattice (red = processing / communication qubits, yellow = recycled environment ancillas).


![IBM Nighthawk square-lattice coupling map with two example QPU partitions highlighted (red = processing / communication qubits, yellow = recycled environment ancillas)](Pics/Nighthawk.png)


## 5 · Remote CNOT Gate — Cat-State Communication (Cat-Comm) Protocol

**Reproduces Fig. 5(a) and Fig. 5(b)**

The Cat-Comm protocol (Section 2.A, Fig. 4a) generates a distributed cat-like entangled
state between communication qubits and uses it to apply a remote CNOT without teleporting
the control qubit's state.  It requires only one shared Bell pair and is the most
resource-efficient protocol for near-term devices.

**Register layout for the 6-qubit circuit** (consistent with QPU layout above):

```
q[0] = CommA      → physical qubit  3   (communication, QPU A)
q[1] = ENA        → physical qubit 14   (environment, QPU A — dashed lines in Fig. 4)
q[2] = Processing → physical qubit  0   (control qubit, QPU A)
q[3] = CommB      → physical qubit  4   (communication, QPU B)
q[4] = ENB        → physical qubit 2   (environment, QPU B — dashed lines in Fig. 4)
q[5] = Processing → physical qubit  7   (target qubit, QPU B)
```

Step 0 is the **monolithic baseline** 'M' (no inter-QPU noise); steps 1–10 each add
a 10 m fiber segment.  The initial ~30 % drop at step 1 is dominated by transducer noise
(κ_T = 0.5); subsequent decay follows the Bell-pair fidelity formula Eq. (3).

![Cat-Comm and TP1 remote CNOT circuits](Pics/Simulation.png)


In [ ]:
# ── Simulator flag ────────────────────────────────────────────────────────────
# True  → all jobs run on AerSimulator (no IBM account / quota needed)
# False → hardware jobs submitted to ibm_torino
simulator = False

# Build backend lists: first 3 entries = hardware (or sim), 4th = AerSimulator reference
_aer = AerSimulator.from_backend(backend)
backends_hw  = [backend, backend, backend, _aer]
backends_sim = [_aer, _aer, _aer, _aer]

results_cat_all = []   # one results-dict per initial state

for state in [0, 1]:
    results = {ft: [] for ft in fibers}

    for i, (fiber_type, kappa_F) in enumerate(fibers.items()):
        my_backend = (backends_sim if simulator else backends_hw)[i]
        sampler    = SamplerV2(mode=my_backend)
        results_cat = []

        for step in range(steps + 1):   # step 0 = monolithic baseline 'M'

            # ── Quantum register — layout matches QPU.Get_Initial_Layout ──────
            # q[0]=CommA(3)  q[1]=ENA(14)  q[2]=Proc_A(0)
            # q[3]=CommB(4)  q[4]=ENB(2)  q[5]=Proc_B(7)
            q = QuantumRegister(6, name='q')
            c = ClassicalRegister(2, name='c')
            qc = QuantumCircuit(q, c)

            # Control qubit is q[2] (physical 0, processing qubit of QPU A)
            if state == 1:
                qc.x(q[2])

            if step == 0:
                # ── Monolithic: direct local CNOT, no inter-QPU communication ─
                qc.cx(q[2], q[5])
            else:
                # ── Distributed: noisy remote CNOT via Cat-Comm ───────────────
                # control = q[2] (Proc A, physical 0)
                # target  = q[5] (Proc B, physical 7)
                # CommA   = q[0] (physical 3)
                # ENA     = q[1] (physical 14)
                # CommB   = q[3] (physical 4)
                # ENB     = q[4] (physical 2)
                RemoteGates.remote_cx(
                    qc,
                    control=q[2], target=q[5],
                    CommA=q[0],   ENA=q[1],
                    CommB=q[3],   ENB=q[4],
                    creg=c,       creg_index=0,
                    kappa_Fiber=kappa_F,
                    Steps=step - 1,        # step 1 → 0 extra collisions
                    kappa_Transductor=kappa_Transductor,
                )

            # Measure processing qubits
            qc.measure(q[2], c[0])   # Proc A (physical 0) → c[0]
            qc.measure(q[5], c[1])   # Proc B (physical 7) → c[1]

            # ── Apply initial_layout so transpiler respects physical qubit ─────
            initial_layout = QPU.Get_Initial_Layout(QPUs, QRG=q)
            pm = generate_preset_pass_manager(
                optimization_level=3,
                target=my_backend.target,
                initial_layout=initial_layout,
            )
            transpiled = pm.run(qc)
            result     = sampler.run([transpiled], shots=shots)
            print(f"  [{fiber_type}  |{state}⟩  step={step:2d}]  job={result.job_id()}")

            data   = result.result()[0].data
            attr   = next(iter(vars(data)))
            counts = getattr(data, attr).get_counts()

            # Extract the two relevant qubits (c[1] c[0] → positions 1, 0)
            bit_counts = defaultdict(int)
            for bitstring, count in counts.items():
                extracted = ''.join(bitstring[p] for p in (1, 0))
                bit_counts[extracted] += count

            results_cat.append(dict(bit_counts))

        # Success: both processing qubits agree — '00' for state=0, '11' for state=1
        expected = f'{state}{state}'
        results[fiber_type] = [entry.get(expected, 0) for entry in results_cat]

    results_cat_all.append(results.copy())

    # ── Plot — reproduces Fig. 5(a) [state=0] or Fig. 5(b) [state=1] ─────────
    Representation.plot_normalized_results(
        results,
        plot_type="Cat-Comm",
        state=f"|{state}\rangle",
        shots=shots,
        kappa_T=kappa_Transductor,
    )


## 6 · Remote CNOT Gate — Teleportation Protocol TP1

**Reproduces Fig. 5(c) and Fig. 5(d)**

TP1 (Section 2.A, Fig. 4b) teleports the entire control-qubit state from QPU A to
the communication qubit of QPU B, where the CNOT is applied locally.  Unlike Cat-Comm,
TP1 collapses the control qubit at its original location; a second teleportation would
be required to restore it (TP2 / TP-Safe).  TP1 is included to demonstrate that the
framework supports multiple remote-gate protocols.

**Register layout** is identical to Section 5:

```
q[0]=CommA(3)  q[1]=ENA(14)  q[2]=Processing_A(0)   ← QPU A
q[3]=CommB(4)  q[4]=ENB(2)  q[5]=Processing_B(7)   ← QPU B
```

After TP1 the teleported control state resides in **CommB = q[3]** (physical qubit 4).
The measurement therefore reads `q[3]` (teleported control) and `q[5]` (target).


In [ ]:
results_tp1_all = []

for state in [0, 1]:
    results = {ft: [] for ft in fibers}

    for i, (fiber_type, kappa_F) in enumerate(fibers.items()):
        my_backend = (backends_sim if simulator else backends_hw)[i]
        sampler    = SamplerV2(mode=my_backend)
        results_tp1 = []

        for step in range(steps + 1):   # step 0 = monolithic 'M'

            q = QuantumRegister(6, name='q')
            c = ClassicalRegister(2, name='c')
            qc = QuantumCircuit(q, c)

            # Control qubit: q[2] (physical 0, Proc A)
            if state == 1:
                qc.x(q[2])

            if step == 0:
                # ── Monolithic baseline ───────────────────────────────────────
                qc.cx(q[2], q[5])
            else:
                # ── Distributed: noisy remote CNOT via TP1 ───────────────────
                RemoteGates.remote_cx_TP1(
                    qc,
                    control=q[2], target=q[5],
                    CommA=q[0],   ENA=q[1],
                    CommB=q[3],   ENB=q[4],
                    creg=c,       creg_index=0,
                    kappa_Fiber=kappa_F,
                    Steps=step - 1,
                    kappa_Transductor=kappa_Transductor,
                )

            # In TP1 the control state is teleported to CommB = q[3] (physical 4)
            # Measure: q[3] (teleported control, CommB) and q[5] (target, Proc B)
            qc.measure(q[3], c[0])   # CommB (physical 4) → c[0]
            qc.measure(q[5], c[1])   # Proc B (physical 7) → c[1]

            initial_layout = QPU.Get_Initial_Layout(QPUs, QRG=q)
            pm = generate_preset_pass_manager(
                optimization_level=3,
                target=my_backend.target,
                initial_layout=initial_layout,
            )
            transpiled = pm.run(qc)
            result     = sampler.run([transpiled], shots=shots)
            print(f"  [{fiber_type}  |{state}⟩  step={step:2d}]  job={result.job_id()}")

            data   = result.result()[0].data
            attr   = next(iter(vars(data)))
            counts = getattr(data, attr).get_counts()

            bit_counts = defaultdict(int)
            for bitstring, count in counts.items():
                extracted = ''.join(bitstring[p] for p in (1, 0))
                bit_counts[extracted] += count

            results_tp1.append(dict(bit_counts))

        expected = f'{state}{state}'
        results[fiber_type] = [entry.get(expected, 0) for entry in results_tp1]

    results_tp1_all.append(results.copy())

    # ── Plot — reproduces Fig. 5(c) [state=0] or Fig. 5(d) [state=1] ─────────
    Representation.plot_normalized_results(
        results,
        plot_type="TP1",
        state=f"|{state}\rangle",
        shots=shots,
        kappa_T=kappa_Transductor,
    )


## 7 · Distributed Grover's Search Algorithm

**Reproduces Fig. 6**

The 2-qubit Grover's search algorithm (Section 2.C, Fig. 7) searches over
{|00⟩, |01⟩, |10⟩, |11⟩}.  For one marked state a single Grover iteration maximises
the success probability.

**6-qubit distributed register layout** — identical to Sections 5 & 6:

```
q[0]=CommA(3)  q[1]=ENA(14)  q[2]=QPUA(0)   ← QPU A (processing qubit = q[2])
q[3]=CommB(4)  q[4]=ENB(2)  q[5]=QPUB(7)   ← QPU B (processing qubit = q[5])
```

Inside `grover_2qubit_annotated_Distributed`, the register is constructed as:

```python
QPUA, ENA, CommA, CommB, ENB, QPUB = q   # q[0..5]
```

So the Grover circuit already uses `q[0]=QPUA` internally.  The `Get_Initial_Layout`
call maps this as:
- `q[0]` (QPUA) → must map to Comm of QPU A = **3**

Wait — this ordering differs from the remote-gate circuits.  The Grover circuit's
internal register order is **QPUA, ENA, CommA, CommB, ENB, QPUB**, which means:
- `q[0]` = QPUA (processing) → maps to `QPUA_layout.Comm = 3` via `Get_Initial_Layout`

This is a known mismatch in the original code.  The fix is to pass a custom
`grover_layout` that correctly assigns the Grover register to physical qubits:

| Grover register | Physical qubit | Role |
|---|---|---|
| `q[0]` = QPUA | **0** | Processing A |
| `q[1]` = ENA | **14** | Environment A |
| `q[2]` = CommA | **3** | Communication A |
| `q[3]` = CommB | **4** | Communication B |
| `q[4]` = ENB | **2** | Environment B |
| `q[5]` = QPUB | **7** | Processing B |

Communication uses **G-654-E** fiber (α = 0.0392 km⁻¹, lowest attenuation) and
κ_T = 0.5.  The first distributed step (~20 m effective) is compared with the
ion-trap result of Main *et al.*, Nature 638, 383–388 (2025) [ref 18] (~71 %).

![Monolithic and distributed Grover circuits](Pics/Grover.png)


In [ ]:
# κ_F for G-654-E fiber (lowest attenuation — used for Grover and QFT)
kappa_F_grover = fibers['G-654-E']

# Ion-trap experimental reference (Main et al., Nature 2025 [18]):
# ~71 % success at step 1 (2 m SMF ≈ first distributed step in this model)
iontrap_ref = 0.71

# ── Custom initial layout for the Grover 6-qubit register ────────────────────
# The Grover circuit uses register order: QPUA, ENA, CommA, CommB, ENB, QPUB
# Physical assignments:
#   q[0]=QPUA   → 0   (processing A)
#   q[1]=ENA    → 14  (environment A)
#   q[2]=CommA  → 3   (communication A)
#   q[3]=CommB  → 4   (communication B)
#   q[4]=ENB    → 2  (environment B)
#   q[5]=QPUB   → 7   (processing B)
def make_grover_layout(qreg):
    """Build Layout for the Grover 6-qubit register."""
    phys = [0, 14, 3, 4, 2, 7]   # physical qubit for q[0]..q[5]
    return Layout({qreg[i]: phys[i] for i in range(6)})

# All four 2-bit marked states
all_marked    = [['00'], ['11'], ['01'], ['10']]
grover_results = {}   # key = marked-state string, value = list length (steps+1)

for marked_states in all_marked:
    label = marked_states[0]
    probs = []

    for step in range(steps + 1):   # step 0 = monolithic 'M'

        my_backend = _aer if simulator else backend
        sampler    = SamplerV2(mode=my_backend)

        if step == 0:
            # ── Monolithic 2-qubit Grover (Fig. 7a) ──────────────────────────
            q2  = QuantumRegister(2, 'q')
            c2  = ClassicalRegister(2, 'c')
            qc  = QuantumCircuit(q2, c2)
            qc.h([q2[0], q2[1]])
            # Oracle
            if label[0] == '0': qc.x(q2[0])
            if label[1] == '0': qc.x(q2[1])
            qc.cz(q2[0], q2[1])
            if label[0] == '0': qc.x(q2[0])
            if label[1] == '0': qc.x(q2[1])
            # Diffusion
            qc.h([q2[0], q2[1]])
            qc.x([q2[0], q2[1]])
            qc.h(q2[1]); qc.cx(q2[0], q2[1]); qc.h(q2[1])
            qc.x([q2[0], q2[1]])
            qc.h([q2[0], q2[1]])
            qc.measure(q2[0], c2[0])
            qc.measure(q2[1], c2[1])

            # Map monolithic qubits to processing qubits (physical 0 and 7)
            mono_layout = Layout({q2[0]: 0, q2[1]: 7})
            pm = generate_preset_pass_manager(
                optimization_level=3,
                target=my_backend.target,
                initial_layout=mono_layout,
            )
        else:
            # ── Distributed Grover (Fig. 7b) ──────────────────────────────────
            # grover_2qubit_annotated_Distributed builds register internally:
            # q[0]=QPUA, q[1]=ENA, q[2]=CommA, q[3]=CommB, q[4]=ENB, q[5]=QPUB
            qc = Algorithms.grover_2qubit_annotated_Distributed(
                marked_states,
                kappa_Fiber=kappa_F_grover,
                Steps=step - 1,
                kappa_Transductor=kappa_Transductor,
            )
            # Apply the Grover-specific initial layout
            grover_layout = make_grover_layout(qc.qregs[0])
            pm = generate_preset_pass_manager(
                optimization_level=3,
                target=my_backend.target,
                initial_layout=grover_layout,
            )

        transpiled = pm.run(qc)
        result     = sampler.run([transpiled], shots=shots)
        print(f"  [Grover  marked={label}  step={step:2d}]  job={result.job_id()}")

        data   = result.result()[0].data
        attr   = next(iter(vars(data)))
        counts = getattr(data, attr).get_counts()

        # For monolithic: bitstring has 2 bits → check directly
        # For distributed: classical register has 6 bits; result bits are c[4] and c[5]
        # (qc.measure([QPUA, QPUB], [4, 5]) in grover_2qubit_annotated_Distributed)
        success_count = 0
        for bitstring, count in counts.items():
            if step == 0:
                # 2-bit register: c[1]c[0] in Qiskit's reversed notation
                bits = bitstring[-2:][::-1]   # → [c[0], c[1]]
                if bits == label:
                    success_count += count
            else:
                # 6-bit register: c[5]c[4]c[3]c[2]c[1]c[0] in Qiskit notation
                # Measured: QPUA→c[4], QPUB→c[5]
                # Qiskit bitstring is reversed: position 0 = c[5], position 1 = c[4]
                c4 = bitstring[1]   # c[4] = QPUA result
                c5 = bitstring[0]   # c[5] = QPUB result
                if c4 + c5 == label:
                    success_count += count

        probs.append(success_count / shots)

    grover_results[label] = probs
    print(f"  Grover marked={label}: monolithic={probs[0]:.3f}, "
          f"step1={probs[1]:.3f} (ion-trap ref={iontrap_ref})")

# ── Plot — reproduces Fig. 6 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
plt.rcParams.update({'font.family': 'serif', 'font.size': 13})

x_pos    = list(range(steps + 1))
x_labels = ['M'] + list(range(1, steps + 1))
titles   = {'00': 'State 00', '11': 'State 11', '01': 'State 01', '10': 'State 10'}

for ax, (label, probs) in zip(axes, grover_results.items()):
    ax.bar(x_pos, probs, color='steelblue', alpha=0.7, label='This work')
    ax.axhline(iontrap_ref, color='navy', linestyle='--',
               linewidth=1.5, label='Ion-Trap [18]')
    ax.set_xticks(x_pos); ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Fiber Steps")
    ax.set_title(titles[label])
    if ax is axes[0]:
        ax.set_ylabel("Measured Probability")
    ax.legend(fontsize=9)

plt.suptitle(
    f"Distributed Grover's Search  |  G-654-E  (α = 0.0392 km⁻¹)  "
    f"|  $\kappa_T = {kappa_Transductor}$",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.savefig("Grover_results_reproduced.png", dpi=300)
plt.show()


## 8 · Distributed 5-Qubit Quantum Fourier Transform (QFT)

**Reproduces Fig. 9**

The 5-qubit QFT is distributed across QPU A (qubits QA1, QA2) and QPU B (qubits QB1,
QB2, QB3) as described in Section 2.D and illustrated in Fig. 8(c).

**9-qubit distributed register layout:**

```
QPU A side:                               QPU B side:
  q[0]=QA1  → physical  0  (Proc A-1)      q[4]=CommB → physical  4  (Comm B)
  q[1]=QA2  → physical  7  (Proc A-2)      q[5]=ENB   → physical 2  (ENV B)
  q[2]=ENA  → physical 14  (ENV A)          q[6]=QB1   → physical  ?  (Proc B-1)
  q[3]=CommA → physical  3  (Comm A)        q[7]=QB2   → physical  ?  (Proc B-2)
                                            q[8]=QB3   → physical  ?  (Proc B-3)
```

For the QFT experiment we require **5 processing qubits** (2 on QPU A, 3 on QPU B).
The QPU B processing qubits (QB1, QB2, QB3) are taken from qubits adjacent to the
communication qubit on QPU B.  Using the ibm_torino coupling map:
- **CommB = 4** → neighbours in QPU B row: **5, 6, 7**
- QB1 = 5, QB2 = 6, QB3 = 7  (consecutive, good connectivity)

The `initial_layout` for the QFT 9-qubit register maps:

| Register | Physical | Role |
|---|---|---|
| `q[0]` QA1 | **0** | Processing A-1 |
| `q[1]` QA2 | **1** | Processing A-2 (qubit next to 0 in QPU A) |
| `q[2]` ENA | **14** | Environment A |
| `q[3]` CommA | **3** | Communication A |
| `q[4]` CommB | **4** | Communication B |
| `q[5]` ENB | **2** | Environment B |
| `q[6]` QB1 | **5** | Processing B-1 |
| `q[7]` QB2 | **6** | Processing B-2 |
| `q[8]` QB3 | **7** | Processing B-3 |

Since the QFT outputs a superposition, **quantum state tomography** (Uhlmann fidelity)
is used to evaluate performance under noise.

Communication uses **G-654-E** fiber (α = 0.0392 km⁻¹) and κ_T = 0.5.

![Distributed 5-qubit QFT circuit](Pics/QFT-Final.png)


In [ ]:
from qiskit.quantum_info import state_fidelity, Statevector
from qiskit_experiments.library import StateTomography

kappa_F_qft = fibers['G-654-E']   # κ_F from G-654-E (α = 0.0392 km⁻¹)
# kappa_Transductor = 0.5 (global — same as all other experiments)

# ── Physical qubit assignments for QFT 9-qubit register ──────────────────────
#   q[0]=QA1   → 0   (processing A-1)
#   q[1]=QA2   → 1   (processing A-2; qubit 1 is adjacent to 0 in QPU A)
#   q[2]=ENA   → 14  (environment A)
#   q[3]=CommA → 3   (communication A, boundary qubit)
#   q[4]=CommB → 4   (communication B, boundary qubit, direct edge 3↔4)
#   q[5]=ENB   → 2  (environment B)
#   q[6]=QB1   → 5   (processing B-1)
#   q[7]=QB2   → 6   (processing B-2)
#   q[8]=QB3   → 7   (processing B-3)
QFT_PHYS = [0, 1, 14, 3, 4, 2, 5, 6, 7]

def make_qft_layout(qreg):
    """Build Layout for the QFT 9-qubit register."""
    return Layout({qreg[i]: QFT_PHYS[i] for i in range(9)})

# Ideal 5-qubit QFT output state (for fidelity reference)
qft_ideal = Algorithms.qft_circuit(5)
psi_ideal  = Statevector.from_instruction(qft_ideal)

sim_backend = AerSimulator.from_backend(backend)
sampler_sim = SamplerV2(mode=sim_backend)

qft_fidelities_hw  = []   # ibm_torino hardware
qft_fidelities_sim = []   # AerSimulator reference

# ── Monolithic QFT — baseline 'M' ────────────────────────────────────────────
print("Running monolithic QFT tomography ...")
for fid_list, bk in [(qft_fidelities_hw, backend),
                     (qft_fidelities_sim, sim_backend)]:
    start = time.time()
    exp   = StateTomography(qft_ideal, backend=bk)
    job   = exp.run(backend=bk, seed_simulation=100).block_for_results()
    fid   = float(
        job.analysis_results("state_fidelity", dataframe=True).iloc[0].value
    )
    tag   = "hardware" if bk is backend else "simulator"
    print(f"  Monolithic QFT fidelity ({tag}): {fid*100:.1f} %  "
          f"[{time.time()-start:.1f} s]")
    fid_list.append(fid)

# ── Distributed QFT — steps 1 … steps ───────────────────────────────────────
for step in range(1, steps + 1):
    print(f"\nDistributed QFT step {step}/{steps} ...")
    for fid_list, bk, samp in [
        (qft_fidelities_hw,  backend,     SamplerV2(mode=backend)),
        (qft_fidelities_sim, sim_backend, sampler_sim),
    ]:
        start   = time.time()
        qc_dist = Algorithms.qft_5qubit_annotated_Distributed(
            Steps=step - 1,              # step 1 → 0 extra fiber collisions
            kappa_Fiber=kappa_F_qft,
            kappa_Transductor=kappa_Transductor,   # 0.5
        )

        qft_layout = make_qft_layout(qc_dist.qregs[0])
        pm_qft = generate_preset_pass_manager(
            optimization_level=3,
            target=bk.target,
            initial_layout=qft_layout,
        )
        # (StateTomography wraps the circuit; layout is passed via the pass-manager
        # applied to the underlying circuit before running tomography)
        qc_dist_t = pm_qft.run(qc_dist)

        # Tomography on all 9 physical qubits mapped above.
        # measurement_indices selects the 5 *processing* qubits in logical order:
        #   logical q1=QA1(q[0]), q2=QB1(q[6]), q3=QB2(q[7]), q4=QB3(q[8]), q5=QA2(q[1])
        exp = StateTomography(
            qc_dist,
            backend=bk,
            physical_qubits=QFT_PHYS,            # 9 physical qubits
            measurement_indices=[0, 6, 7, 8, 1], # logical order: QA1,QB1,QB2,QB3,QA2
        )
        job = exp.run(sampler=samp, seed_simulation=100).block_for_results()
        rho = job.analysis_results("state").value
        fid = state_fidelity(rho, psi_ideal)

        tag = "hw" if bk is backend else "sim"
        print(f"  [{tag}  step={step}]  fidelity={fid*100:.1f} %  "
              f"[{time.time()-start:.1f} s]")
        fid_list.append(fid)

# ── Plot — reproduces Fig. 9 ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
plt.rcParams.update({'font.family': 'serif', 'font.size': 16})

x      = np.arange(steps + 1)
labels = ['M'] + list(range(1, steps + 1))

ax.plot(x, np.array(qft_fidelities_hw)  * 100,
        marker='o',  linestyle='-',  label='ibm\_torino')
ax.plot(x, np.array(qft_fidelities_sim) * 100,
        marker='s',  linestyle='--', label='AerSimulator')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Fiber Steps")
ax.set_ylabel("Fidelity (%)")
ax.set_title(
    f"5-Qubit Distributed QFT  |  G-654-E  "
    f"(α = 0.0392 km⁻¹)  |  $\kappa_T = {kappa_Transductor}$",
    fontsize=13,
)
ax.legend()
plt.tight_layout()
plt.savefig("QFT_fidelity_reproduced.png", dpi=300)
plt.show()
plt.close()

print(f"\nMonolithic fidelity (hardware) : {qft_fidelities_hw[0]*100:.1f} %")
print(f"Min distributed fidelity (hw)  : {min(qft_fidelities_hw[1:])*100:.1f} %")


## 9 · Summary of Results

| Experiment | Key Observation | Paper Figure |
|---|---|---|
| Remote CNOT Cat-Comm \|0⟩ | ~30 % initial drop (transducer); further decay with fiber steps | Fig. 5(a) |
| Remote CNOT Cat-Comm \|1⟩ | Higher noise sensitivity due to extra X gate + T₁ relaxation | Fig. 5(b) |
| Remote CNOT TP1 \|0⟩ | Slightly better than Cat-Comm; teleports control to CommB (phys. 4) | Fig. 5(c) |
| Remote CNOT TP1 \|1⟩ | Similar trend to Cat-Comm \|1⟩ | Fig. 5(d) |
| Distributed Grover's search | Step-1 result ≈ 71 % matches ion-trap experiment (Main et al. 2025) | Fig. 6 |
| Distributed 5-qubit QFT | Monolithic ~63 %; dominant drop at transduction; total drop to ~40 % | Fig. 9 |

**Physical qubit summary across all experiments:**

| Role | Physical qubit | Used in |
|---|---|---|
| Processing A | 0 | All remote-gate, Grover, QFT |
| Communication A | 3 | All remote-gate, Grover, QFT |
| Environment A | 14 | All remote-gate, Grover, QFT |
| Communication B | 4 | All remote-gate, Grover, QFT |
| Environment B | 2 | All remote-gate, Grover, QFT |
| Processing B | 7 | All remote-gate, Grover |
| Processing A-2 (QA2) | 1 | QFT only |
| Processing B-1 (QB1) | 5 | QFT only |
| Processing B-2 (QB2) | 6 | QFT only |
| Processing B-3 (QB3) | 7 | QFT only |
